# Fatores Socioeconômicos Associados ao Desempenho no ENEM 2022

Análise exploratória interativa sobre a base tratada gerada pelo pipeline em `src/`.

**Pré-requisito:** execute antes, na pasta `src/`:
```
python 02_criar_amostra.py
python 03_tratar_dados.py
```

> Cuidado interpretativo: todas as diferenças aqui são **associações** observadas em uma amostra — não indicam causalidade.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJETO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABELAS = PROJETO / "outputs" / "tabelas"

df = pd.read_csv(TABELAS / "enem_2022_tratado.csv", dtype={"NU_INSCRICAO": "string"})
presenca = pd.read_csv(TABELAS / "enem_2022_base_presenca.csv", dtype={"NU_INSCRICAO": "string"})
print(f"Base de desempenho: {len(df):,} participantes")
print(f"Base de presença:   {len(presenca):,} inscritos (não treineiros)")
df.head(3)

## 1. Distribuição geral das notas

In [ ]:
notas = ["NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT", "NU_NOTA_REDACAO", "media_geral"]
df[notas].describe().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(df["media_geral"], bins=60, color="#2a78d6")
ax.set_title("Distribuição da média geral")
ax.set_xlabel("Média geral (5 provas)")
ax.set_ylabel("Participantes")
plt.show()

## 2. Média por tipo de escola e por renda familiar

In [ ]:
df.groupby("tipo_escola")["media_geral"].agg(["count", "mean"]).round(2).sort_values("mean", ascending=False)

In [ ]:
media_renda = df.groupby("renda_familiar")["media_geral"].mean().sort_index()
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(media_renda.index, media_renda.values, color="#2a78d6")
ax.set_title("Média geral por faixa de renda familiar")
ax.set_ylabel("Média geral")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 3. Escolaridade dos pais, internet e computador

In [ ]:
print("Escolaridade da mãe:")
display(df.groupby("escolaridade_mae")["media_geral"].agg(["count", "mean"]).round(2))
print("Internet em casa:")
display(df.groupby("internet_casa")["media_geral"].agg(["count", "mean"]).round(2))
print("Computador em casa:")
display(df.groupby("computador_casa")["media_geral"].agg(["count", "mean"]).round(2))

## 4. UFs e taxa de presença

In [ ]:
df.groupby("SG_UF_PROVA")["media_geral"].agg(["count", "mean"]).round(2).sort_values("mean", ascending=False).head(10)

In [ ]:
taxa = presenca.groupby("renda_familiar")["presente_todas"].mean().mul(100).round(2).sort_index()
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(taxa.index, taxa.values, color="#2a78d6")
ax.set_ylim(0, 100)
ax.set_title("Taxa de presença nas 4 provas por renda familiar (%)")
ax.set_ylabel("Presentes em todas as provas (%)")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Conclusões rápidas

- A média geral cresce quase monotonicamente com a renda familiar.
- Escola privada supera a pública em ~95 pontos; a diferença é maior em Redação e Matemática.
- Escolaridade dos pais, internet e computador em casa acompanham o mesmo gradiente.
- A **ausência** às provas também é maior nas faixas de renda mais baixas — a desigualdade aparece antes da nota.

Ver `outputs/relatorio_final.md`, `docs/conclusoes.md` e `docs/limitacoes_do_estudo.md`.